In [6]:
import os
import glob
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import json
from pathlib import Path
import numpy as np

In [11]:
# Base directory containing all expression runs
base_dir = "/mnt/10tb/home/vsfishman/DNALM/runs/expression"

# Function to extract scores from TensorBoard event files
def extract_scores_from_events(event_file_path, run_name):
	"""Extract all scalar values at a specific step from TensorBoard event file."""
	try:
		ea = EventAccumulator(event_file_path)
		ea.Reload()
		
		scores = []
		for tag in ea.Tags()['scalars']:
			events = ea.Scalars(tag)
			# Find events at or closest to target step
			step_values = [(event.step, event.value) for event in events]
						
			for step, value in step_values:
				if step > 100: # skip test runs
					scores.append({"step": step, "value": value, "run_name": run_name, "metric": tag})
		return scores
	except Exception as e:
		print(f"Error processing {event_file_path}: {e}")
		return {}

# Main function to traverse all subdirectories and collect scores
def collect_all_scores(base_dir):
	"""Traverse all subdirectories and collect scores at specified step."""
	all_results = []
	
	# Get all subdirectories
	subdirs = [d for d in glob.glob(os.path.join(base_dir, "*")) if os.path.isdir(d)]
	
	print(f"Found {len(subdirs)} subdirectories to process")
	
	for run_dir in subdirs:
		run_name = os.path.basename(run_dir)
		print(f"Processing: {run_name}")
		
		# Find TensorBoard event files
		event_files = glob.glob(os.path.join(run_dir, "events.out.tfevents*"))
		
		if not event_files:
			print(f"  No event files found in {run_name}")
			continue
		
		# Get config information
		# config_info = get_config_info(run_dir)
		
		# Process each event file
		for event_file in event_files:
			scores = extract_scores_from_events(event_file, run_name=run_dir)
			
			if scores:
				all_results.extend(scores)
				print(f"  Found {len(scores)} metrics")
			else:
				print(f"  No scores found")
	
	return all_results

# Execute the collection
print(f"Starting to collect scores from all subdirectories...")
results = collect_all_scores(base_dir)

print(f"\nCollected data from {len(results)} runs")

# Convert to DataFrame for easier analysis
if results:
	df = pd.DataFrame.from_records(results)
	print(f"\nDataFrame shape: {df.shape}")
	print("\nColumns:", df.columns.tolist())
	
	# Display first few rows
	print("\nFirst few results:")
	display(df.head())
	
	# Save to CSV
	output_file = f"expression_scores.csv"
	df.to_csv(output_file, index=False)
	print(f"\nResults saved to {output_file}")
else:
	print("No results found!")

Starting to collect scores from all subdirectories...
Found 84 subdirectories to process
Processing: 20250417-001905
  No event files found in 20250417-001905
Processing: 20250415-000104
  No scores found
Processing: 20250415-000429
  No event files found in 20250415-000429
Processing: 20250413-182651
  No event files found in 20250413-182651
Processing: 20250420-214247
  No scores found
Processing: 20250414-234356
  No event files found in 20250414-234356
Processing: 20250420-211327
  No scores found
Processing: 20250413-184108
  No event files found in 20250413-184108
Processing: 20250420-205816
  No scores found
Processing: 20250413-202948
  Found 68 metrics
Processing: 20250419-222413
  Found 56 metrics
Processing: 20250420-214407
  No scores found
Processing: 20250417-003058
  No event files found in 20250417-003058
Processing: 20250413-194414
  No scores found
Processing: 20250413-214943
  No scores found
Processing: 20250420-211114
  No scores found
Processing: 20250417-161117
 

,step,value,run_name,metric
0,250,11.555820,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
1,500,11.434798,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
2,750,11.261512,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
3,1000,11.167288,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
4,1250,10.246181,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train



Results saved to expression_scores.csv


In [12]:
df["run_name"].unique()

array(['/mnt/10tb/home/vsfishman/DNALM/runs/expression/20250413-202948',
       '/mnt/10tb/home/vsfishman/DNALM/runs/expression/20250419-222413',
       '/mnt/10tb/home/vsfishman/DNALM/runs/expression/20250413-220701'],
      dtype=object)

In [13]:
df.query("run_name == '/mnt/10tb/home/vsfishman/DNALM/runs/expression/20250413-202948'")

,step,value,run_name,metric
0,250,11.555820,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
1,500,11.434798,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
2,750,11.261512,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
3,1000,11.167288,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
4,1250,10.246181,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
...,...,...,...,...
63,1000,0.000000,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,patience/samples
64,2000,1.000000,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,patience/samples
65,3000,0.000000,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,patience/samples
66,4000,1.000000,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,patience/samples


In [24]:
_ = df.query("run_name == '/mnt/10tb/home/vsfishman/DNALM/runs/expression/20250413-202948'")
_[_.metric.str.contains("iterations")].head(2)

,step,value,run_name,metric
0,250,11.555820,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
1,500,11.434798,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train


In [25]:
_[_.metric.str.contains("samples")].head(2)

,step,value,run_name,metric
6,1000,11.555820,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/samples/train
7,2000,11.434798,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/samples/train


In [ ]:
# Analyze the collected scores
if 'df' in locals() and not df.empty:
	print("=== Score Analysis ===")
	
	# Get all metric columns (exclude metadata columns)
	metadata_cols = ['run_name', 'run_path', 'target_step', 'model_name', 'learning_rate', 'batch_size', 'num_training_steps']
	metric_cols = [col for col in df.columns if col not in metadata_cols]
	
	print(f"\nFound {len(metric_cols)} metrics:")
	for col in metric_cols:
		print(f"  - {col}")
	
	# Show summary statistics for each metric
	for metric in metric_cols:
		print(f"\n=== {metric} ===")
		# Extract just the values (not the dict structure)
		values = []
		for val in df[metric]:
			if isinstance(val, dict) and 'value' in val:
				values.append(val['value'])
			elif isinstance(val, (int, float)):
				values.append(val)
		
		if values:
			values = np.array(values)
			print(f"Count: {len(values)}")
			print(f"Mean: {values.mean():.4f}")
			print(f"Std: {values.std():.4f}")
			print(f"Min: {values.min():.4f}")
			print(f"Max: {values.max():.4f}")
		else:
			print("No valid values found")
	
	# Show runs with best performance for common metrics
	common_metrics = ['loss', 'accuracy', 'f1', 'precision', 'recall', 'mse', 'mae']
	
	for metric in common_metrics:
		matching_cols = [col for col in metric_cols if metric.lower() in col.lower()]
		if matching_cols:
			print(f"\n=== Best {metric} scores ===")
			for col in matching_cols:
				# Find best value
				best_idx = None
				best_val = None
				
				for idx, val in enumerate(df[col]):
					if isinstance(val, dict) and 'value' in val:
						current_val = val['value']
					elif isinstance(val, (int, float)):
						current_val = val
					else:
						continue
					
					if best_val is None or current_val > best_val:  # Assuming higher is better
						best_val = current_val
						best_idx = idx
				
				if best_idx is not None:
					print(f"{col}: {best_val:.4f} (Run: {df.iloc[best_idx]['run_name']})")

=== Score Analysis ===

Found 74 metrics:
  - loss/iterations/train
  - loss/samples/train
  - time/iterations/per_iter
  - time/samples/per_iter
  - lr/iterations/param_group_0
  - lr/samples/param_group_0
  - gradients_global_norm/iterations
  - gradients_global_norm/samples
  - loss/iterations/valid
  - loss/samples/valid
  - patience/iterations
  - patience/samples
  - pearson_corr/iterations/train
  - pearson_corr/samples/train
  - pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/iterations/train
  - pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/train
  - pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/iterations/train
  - pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/train
  - pearson_corr_5_cells_borzoi_cells_Expression_dataset_v1_mm10_CPM dataset/iterations/train
  - pearson_corr_5_cells_borzoi_cells_Expression_dataset_v1_mm10_CPM dataset/samples/train
  - pearson_corr_5_cells_borzoi_genes_Expression_dataset_v1_mm10_CPM

In [ ]:
# NEW: Extract ALL steps and metrics for ALL models
print("=== Creating comprehensive dataframe with all models, all steps, all metrics ===")

# Function to extract all scores from TensorBoard event files
def extract_all_scores_from_events(event_file_path):
	"""Extract all scalar values at all steps from TensorBoard event file."""
	try:
		ea = EventAccumulator(event_file_path)
		ea.Reload()
		
		all_scores = []
		for tag in ea.Tags()['scalars']:
			events = ea.Scalars(tag)
			for event in events:
				all_scores.append({
					'metric': tag,
					'step': event.step,
					'value': event.value,
					'wall_time': event.wall_time
				})
		
		return all_scores
	except Exception as e:
		print(f"Error processing {event_file_path}: {e}")
		return []

# Function to get config information
def get_config_info(run_dir):
	"""Extract relevant information from config files."""
	config_info = {}
	
	# Try to read config.json
	config_json_path = os.path.join(run_dir, "config.json")
	if os.path.exists(config_json_path):
		try:
			with open(config_json_path, 'r') as f:
				config = json.load(f)
				config_info.update({
					'model_name': config.get('model_name', 'unknown'),
					'learning_rate': config.get('learning_rate', 'unknown'),
					'batch_size': config.get('batch_size', 'unknown'),
					'num_training_steps': config.get('num_training_steps', 'unknown')
				})
		except Exception as e:
			print(f"Error reading config.json in {run_dir}: {e}")
	
	return config_info

# Main function to collect ALL data
def collect_all_data_comprehensive(base_dir):
	"""Collect ALL data from ALL runs - all steps, all metrics."""
	all_results = []
	
	# Get all subdirectories
	subdirs = [d for d in glob.glob(os.path.join(base_dir, "*")) if os.path.isdir(d)]
	
	print(f"Found {len(subdirs)} subdirectories to process")
	
	for run_dir in subdirs:
		run_name = os.path.basename(run_dir)
		print(f"Processing: {run_name}")
		
		# Find TensorBoard event files
		event_files = glob.glob(os.path.join(run_dir, "events.out.tfevents*"))
		
		if not event_files:
			print(f"  No event files found in {run_name}")
			continue
		
		# Get config information
		config_info = get_config_info(run_dir)
		
		# Process each event file
		for event_file in event_files:
			scores = extract_all_scores_from_events(event_file)
			
			if scores:
				# Add metadata to each score entry
				for score in scores:
					score.update({
						'run_name': run_name,
						'run_path': run_dir,
						**config_info
					})
				
				all_results.extend(scores)
				print(f"  Found {len(scores)} data points")
			else:
				print(f"  No data found")
	
	return all_results

# Execute the comprehensive collection
print("Starting comprehensive data collection...")
all_data = collect_all_data_comprehensive(base_dir)

print(f"\nCollected {len(all_data)} total data points")

# Convert to DataFrame
if all_data:
	comprehensive_df = pd.DataFrame(all_data)
	print(f"\nComprehensive DataFrame shape: {comprehensive_df.shape}")
	print("\nColumns:", comprehensive_df.columns.tolist())
	
	# Display first few rows
	print("\nFirst few results:")
	display(comprehensive_df.head(10))
	
	# Show summary statistics
	print(f"\n=== Summary Statistics ===")
	print(f"Total runs: {comprehensive_df['run_name'].nunique()}")
	print(f"Total metrics: {comprehensive_df['metric'].nunique()}")
	print(f"Total steps: {comprehensive_df['step'].nunique()}")
	print(f"Step range: {comprehensive_df['step'].min()} to {comprehensive_df['step'].max()}")
	
	# Show unique metrics
	print(f"\n=== Unique Metrics ===")
	unique_metrics = comprehensive_df['metric'].unique()
	for i, metric in enumerate(sorted(unique_metrics)):
		print(f"{i+1:2d}. {metric}")
	
	# Show unique runs
	print(f"\n=== Unique Runs ===")
	unique_runs = comprehensive_df['run_name'].unique()
	for i, run in enumerate(sorted(unique_runs)):
		print(f"{i+1:2d}. {run}")
	
	# Save to CSV
	output_file = "expression_comprehensive_data.csv"
	comprehensive_df.to_csv(output_file, index=False)
	print(f"\nComprehensive data saved to {output_file}")
	
	# Create pivot table for easier analysis
	print(f"\n=== Creating pivot table ===")
	# Pivot to have runs as rows, metrics as columns, and values as cell values
	# We'll use the latest step for each run-metric combination
	pivot_df = comprehensive_df.groupby(['run_name', 'metric'])['value'].last().unstack(fill_value=np.nan)
	
	print(f"Pivot table shape: {pivot_df.shape}")
	print("\nPivot table (first few rows):")
	display(pivot_df.head())
	
	# Save pivot table
	pivot_output_file = "expression_pivot_table.csv"
	pivot_df.to_csv(pivot_output_file)
	print(f"Pivot table saved to {pivot_output_file}")
	
else:
	print("No data found!")
